# Lab 3 — Forecasting baselines and causal temporal features

**Audience:** beginner Strategic Engineering MSc students  
**Scope:** Lecture 3, **slides 1–68 only**  
**Estimated lab time:** 80–90 minutes

## Engineering problem
You are supporting the energy manager of an industrial facility. Every hour, you must forecast the facility's **electricity demand for the next hour**.

A useful forecasting method should not merely have a low error: it should beat credible simple alternatives under the **same chronological evaluation protocol**.

## Learning objectives
By the end of the lab you should be able to:

1. implement constant, persistence, drift, seasonal-naïve, moving-average, and exponential-smoothing baselines;
2. compare methods with MAE and RMSE using a chronological train/validation/test split;
3. transform a time series into a causal tabular feature set using lags, differences, rolling statistics, and contextual variables;
4. fit a simple linear forecasting model on those features;
5. explain what a strong baseline tells you about the structure of the forecasting problem;
6. detect and avoid **future-information leakage** in feature construction.

> **Prediction-time rule:** to predict demand at hour *t*, every signal-derived input must come from times earlier than *t*. A contextual variable may refer to *t* only if it is genuinely known in advance (for example, a production plan).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(7)
plt.rcParams["figure.figsize"] = (10, 4)

## 1. Inspect the engineering data

In [ ]:
# Synthetic but realistic hourly facility data: no download is required.
N_DAYS = 120
N = 24 * N_DAYS
t = np.arange(N)
time = pd.date_range("2026-01-01", periods=N, freq="h")
hour = time.hour.to_numpy()
dow = time.dayofweek.to_numpy()
weekday = (dow < 5).astype(float)

# Ambient temperature with daily + slow variation.
temperature = (
    17
    + 6 * np.sin(2*np.pi*(hour - 8)/24)
    + 2.5 * np.sin(2*np.pi*t/(24*30))
    + np.random.normal(0, 0.8, N)
)

# Planned production intensity: known in advance from the schedule.
working_hours = ((hour >= 7) & (hour <= 18)).astype(float)
production_plan = weekday * (0.35 + 0.65 * working_hours)

# Electricity demand. Temperature affects demand with about a 2-hour delay.
temp_lag2 = np.r_[temperature[:2], temperature[:-2]]
daily = 7 * np.sin(2*np.pi*(hour - 6)/24) + 3*np.cos(4*np.pi*hour/24)
weekly = 3 * np.sin(2*np.pi*t/(24*7))
slow_drift = 0.004 * t
noise = np.random.normal(0, 2.2, N)

demand = (
    72
    + 24 * production_plan
    + 0.55 * temp_lag2
    + daily
    + weekly
    + slow_drift
    + noise
)

df = pd.DataFrame({
    "demand": demand,
    "temperature": temperature,
    "production_plan": production_plan,
}, index=time)

df.head()

In [ ]:
ax = df.iloc[:24*14][["demand", "temperature"]].plot(secondary_y="temperature")
ax.set_title("First two weeks: demand and temperature")
ax.set_xlabel("Time")
plt.show()

### Before coding
The target of row *t* is `demand[t]`. Imagine that the forecast is issued just before hour *t* begins.

Classify each candidate input as **available** or **not available** at that moment:
- `demand[t]`
- `demand[t-1]`
- `temperature[t-1]`
- planned production for hour `t`

The key question is always operational: *would this information exist at prediction time?*

## 2. Chronological split and common metrics

In [ ]:
# Chronological split: first 60% train, next 20% validation, last 20% test.
n = len(df)
i_train = int(0.60 * n)
i_val = int(0.80 * n)

train = df.iloc[:i_train].copy()
val = df.iloc[i_train:i_val].copy()
test = df.iloc[i_val:].copy()

print(train.index.min(), "→", train.index.max(), len(train))
print(val.index.min(),   "→", val.index.max(),   len(val))
print(test.index.min(),  "→", test.index.max(),  len(test))

In [ ]:
def metrics(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = np.asarray(y_true)[mask]
    y_pred = np.asarray(y_pred)[mask]
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
    }

def evaluate_period(data, pred_col):
    return metrics(data["demand"].to_numpy(), data[pred_col].to_numpy())

## 3. Baseline ladder: constant, persistence, drift, seasonal naïve

In [ ]:
# Work on one full timeline so that validation/test forecasts may use genuinely
# available observations from immediately before the split boundary.
all_data = df.copy()

# TODO 1 — Constant baseline.
# Fit the constant value on TRAINING data only, then assign it to every row.
train_mean = ...
all_data["pred_mean"] = ...

# TODO 2 — Persistence: predict y[t] with y[t-1].
all_data["pred_persistence"] = ...

# TODO 3 — Drift: y_hat[t] = y[t-1] + (y[t-1] - y[t-2]).
all_data["pred_drift"] = ...

# TODO 4 — Daily seasonal naive for hourly data.
all_data["pred_seasonal24"] = ...

# Re-create chronological views with prediction columns.
train_b = all_data.iloc[:i_train].copy()
val_b   = all_data.iloc[i_train:i_val].copy()
test_b  = all_data.iloc[i_val:].copy()

rows = []
for name in ["pred_mean", "pred_persistence", "pred_drift", "pred_seasonal24"]:
    m = evaluate_period(val_b, name)
    rows.append({"method": name.replace("pred_", ""), **m})

pd.DataFrame(rows).sort_values("MAE")

### Think
Which baseline is strongest on validation? What temporal assumption does each baseline encode?

- constant → target distribution only
- persistence → short-term continuity
- drift → recent local trend continues
- seasonal naïve → known repeating cycle

## 4. Moving-average baseline: select the window on validation

In [ ]:
# Moving-average window length is a model-selection choice.
window_candidates = [3, 6, 12, 24, 48]
ma_results = []

for L in window_candidates:
    col = f"pred_ma_{L}"
    # TODO — Use only observations strictly before target time t.
    # Hint: combine shift(1), rolling(L), and mean().
    all_data[col] = ...
    val_tmp = all_data.iloc[i_train:i_val]
    m = evaluate_period(val_tmp, col)
    ma_results.append({"window": L, **m})

ma_results = pd.DataFrame(ma_results).sort_values("MAE")
ma_results

## 5. Exponential smoothing: select α on validation

In [ ]:
# Exponential smoothing: choose alpha on validation data.
alpha_candidates = [0.05, 0.10, 0.20, 0.40, 0.70, 0.90]
es_results = []

for alpha in alpha_candidates:
    col = f"pred_es_{alpha}"
    # TODO — Compute the exponentially weighted level, then shift it by one step
    # so that the forecast at t does not use y[t].
    all_data[col] = ...
    val_tmp = all_data.iloc[i_train:i_val]
    m = evaluate_period(val_tmp, col)
    es_results.append({"alpha": alpha, **m})

es_results = pd.DataFrame(es_results).sort_values("MAE")
es_results

Recall the recursion:

**x̂ₜ₊₁ = αxₜ + (1 − α)x̂ₜ**

A larger α reacts faster to recent observations; a smaller α produces a smoother, slower estimate.

## 6. Use autocorrelation as a diagnostic probe

In [ ]:
# Autocorrelation is a descriptive probe of repeated temporal structure.
lags = np.arange(1, 49)

# TODO — Compute demand autocorrelation for each lag k.
acf = [... for k in lags]

plt.stem(lags, acf)
plt.axvline(24, linestyle="--", label="24 h")
plt.xlabel("Lag (hours)")
plt.ylabel("Autocorrelation")
plt.title("Training-set demand autocorrelation")
plt.legend()
plt.show()

Look especially at lags near **1 hour** and **24 hours**. A peak at lag 24 is consistent with daily repetition, but autocorrelation is descriptive: it does not establish a causal mechanism.

## 7. Optional: delayed relationship between variables

In [ ]:
# OPTIONAL / guided extension — cross-correlation.
# lag k means correlate demand[t] with temperature[t-k].
xcorr_lags = np.arange(0, 13)

# TODO — complete the correlation calculation.
xcorr = [... for k in xcorr_lags]

plt.stem(xcorr_lags, xcorr)
plt.xlabel("Temperature lag k (hours)")
plt.ylabel("Correlation")
plt.title("Demand[t] vs temperature[t-k] — training data")
plt.show()

Cross-correlation can suggest a useful delay between two recorded variables, but common drivers can create correlation too. Treat it as a feature-design clue, not as proof of causality.

## 8. Build a causal temporal feature table

In [ ]:
# Build a causal feature table for target demand[t].
feat = pd.DataFrame(index=df.index)
feat["target"] = df["demand"]

# TODO — lag features.
feat["lag1"] = ...
feat["lag2"] = ...
feat["lag24"] = ...

# TODO — recent change and a 6-hour endpoint trend.
feat["diff1"] = ...
feat["trend6"] = ...

# TODO — rolling features using history only. Start from a shifted history.
hist = ...
feat["mean6"] = ...
feat["std24"] = ...

# Context / second variable.
feat["temp_lag2"] = ...
feat["production_plan"] = df["production_plan"]  # known in advance

feat = feat.dropna()
feat.head()

The feature table mixes two strategies from the lecture:

- **preserve positions:** explicit lag features such as `lag1`, `lag24`;
- **summarize intervals:** rolling mean, standard deviation, and trend.

It also includes a contextual variable (`production_plan`) that is assumed known at prediction time.

## 9. Linear forecasting model over engineered features

In [ ]:
feature_cols = [c for c in feat.columns if c != "target"]

# Split by timestamps so the feature table follows the same chronology.
train_f = feat.loc[feat.index < val.index.min()]
val_f   = feat.loc[(feat.index >= val.index.min()) & (feat.index < test.index.min())]
test_f  = feat.loc[feat.index >= test.index.min()]

# TODO — fit a LinearRegression on TRAINING rows only.
model = ...
...

# TODO — validation predictions and metrics.
val_pred = ...
print("Feature model validation:", ...)

# Do not inspect the test result until your design choices are fixed.

## 10. Freeze choices and compare once on the test set

In [ ]:
# After all design choices are fixed from validation, evaluate the final methods on test.
best_ma_L = int(ma_results.iloc[0]["window"])
best_alpha = float(es_results.iloc[0]["alpha"])

# TODO — fit the final feature model if you have not already done so, obtain test_pred,
# and build a comparison table containing the selected baselines and feature model.
# Keep exactly the same test interval and metrics for every method.

In [ ]:
# TODO — plot 3–4 days of the test set.
# Compare actual demand with at least two baselines and your feature-based model.

## Interpretation questions
Answer briefly in your own words.

1. Why can persistence be difficult to beat for hourly demand?
2. If the 24-hour seasonal-naïve baseline is strong, what does that tell you about the data?
3. Why do we use `shift(1)` before computing causal rolling features for target `demand[t]`?
4. Why can a planned variable for hour *t* be valid even though actual `demand[t]` is not available?
5. If the engineered feature model improves MAE by only a tiny amount over the best simple baseline, what engineering question should you ask before deploying it?
6. Which manually chosen feature-design decisions could limit generalization?

## Take-away
A forecasting model is useful only relative to a meaningful reference.

The workflow used here is deliberately simple:

**chronological split → credible baselines → validation-based choices → causal features → simple learned model → final test comparison**

This is a strong foundation before moving to more complex temporal models.